In [2]:
import os
import pandas as pd
import numpy as np
import re
import traceback
from datetime import datetime

base_dir  = os.getcwd()

idb = pd.read_excel(os.path.join(base_dir,"IDB_Agrimonitor_-_PSE_Agricultural_Policy_Monitoring_System__data_11182024_Cat_1-3.xlsx"), 
                    sheet_name='IDB_Agrimonitor_-_PSE_Agricultu')

idb = idb[['country', 'code', 'comm_id', 'unit', 'year','Category','value','Commoditie', 'Tstat']]
idb = idb[idb.Category==2]
idb.drop(['Category'], axis=1, inplace=True)
idb.rename(columns={'country': 'COUNTRY_LABEL', 'Commoditie': 'COMMODITY_LABEL', 'comm_id':'COMMODITY_CODE',
                         'unit':'UNIT', 'year':'YEAR', 'value':'VALUE', 'code':'CODE', 'Tstat':'TRADE_STATUS'}, inplace=True)

countries = ['BRAZIL','CANADA','EUROPEAN UNION','COSTA RICA','COLOMBIA','CHILE','MEXICO','UNITED STATES',
             'USA','EU-28','ARGENTINA']
idb = idb[~idb['COUNTRY_LABEL'].isin(countries)]
idb['TRADE_STATUS'] = np.where(idb['TRADE_STATUS']=='M','iMports',
                               np.where(idb['TRADE_STATUS']=='X','eXports', 
                                        np.where(idb['TRADE_STATUS']=='Z','Non Tradable', np.nan)))
idb.head()

,COUNTRY_LABEL,CODE,COMMODITY_CODE,UNIT,YEAR,VALUE,COMMODITY_LABEL,TRADE_STATUS
0,JAMAICA,RP,6,JMD/t,2012,49290.970000,Bananas,Non Tradable
1,HAITI,CT,85,HTGmn,2006,903.280000,Sweet Potatoes,Non Tradable
2,TRINIDAD AND TOBAGO,QP,56,000t,2010,2.310577,Cassava,iMports
3,JAMAICA,QP,34,000t,2006,1745.290000,Refined Sugar,eXports
4,SURINAME,EFC,4,SRDmn,2009,-0.066407,Beef and Veal,iMports


In [3]:
idb_tradestatus = idb[['COUNTRY_LABEL','COMMODITY_LABEL','COMMODITY_CODE','YEAR','TRADE_STATUS']].drop_duplicates()
idb_tradestatus['COMMODITY_LABEL']= idb_tradestatus['COMMODITY_LABEL'].replace('Non MPS Commodities', 'NonMPS from Workbook')
idb_tradestatus.shape

(2342, 5)

In [4]:
variables =['EFC','QC','QP','MPD','MPS','VC','VP','PP','RP', 'PNPC']
idb_sel = idb[idb['CODE'].isin(variables)]

idb_sel = pd.pivot_table(idb_sel, values='VALUE', index=['COUNTRY_LABEL','COMMODITY_CODE', 'COMMODITY_LABEL','YEAR'], 
                     columns=['CODE']).reset_index().rename_axis(None, axis=1)
idb_sel.rename(columns={'QP':'PRODQ', 'QC':'CONSQ','RP':'REFP','PP':'PROP','PNPC':'NPC_SOURCE','MPD':'MPD_SOURCE'}, inplace=True)

idb_sel['COMMODITY_LABEL']= idb_sel['COMMODITY_LABEL'].replace('Non MPS Commodities', 'NonMPS from Workbook')

idb_sel['CONSQ'] = idb_sel['CONSQ']*1000
idb_sel['PRODQ'] = idb_sel['PRODQ']*1000

idb_sel['VC'] = idb_sel['VC']*10e5
idb_sel['VP'] = idb_sel['VP']*10e5
idb_sel['EFC'] = idb_sel['EFC']*10e5
idb_sel['MPS'] = idb_sel['MPS']*10e5

idb_sel['CONSQ_PHY_UNIT'] = np.where(idb_sel['COMMODITY_LABEL']=='NonMPS from Workbook','AG','MT')
idb_sel['PRODQ_PHY_UNIT'] = np.where(idb_sel['COMMODITY_LABEL']=='NonMPS from Workbook','AG','MT')
idb_sel['PROP_PHY_UNIT'] = 'MT'
idb_sel['REFP_PHY_UNIT'] = 'MT'

idb_sel = idb_sel.merge(idb_tradestatus)

idb_sel.head()

,COUNTRY_LABEL,COMMODITY_CODE,COMMODITY_LABEL,YEAR,EFC,MPD_SOURCE,MPS,NPC_SOURCE,PROP,CONSQ,PRODQ,REFP,VC,VP,CONSQ_PHY_UNIT,PRODQ_PHY_UNIT,PROP_PHY_UNIT,REFP_PHY_UNIT,TRADE_STATUS
0,BAHAMAS,2,Avocados,2010,0.0,128.513794,154216.553,1.187012,815.7094,1253.720823,1200.0,687.195606,1022671.860,978851.280,MT,MT,MT,MT,iMports
1,BAHAMAS,2,Avocados,2011,0.0,3.874148,4710.963,1.004772,815.7094,1284.545600,1216.0,811.835253,1047815.920,991902.630,MT,MT,MT,MT,iMports
2,BAHAMAS,2,Avocados,2012,0.0,0.000000,0.000,1.000000,815.7094,1323.609892,1232.0,861.491957,1079681.031,1004953.981,MT,MT,MT,MT,iMports
3,BAHAMAS,2,Avocados,2013,0.0,0.000000,0.000,1.000000,815.7094,1383.172855,1267.0,839.297483,1128267.100,1033503.810,MT,MT,MT,MT,iMports
4,BAHAMAS,2,Avocados,2014,0.0,583.441453,758473.888,1.724350,1388.9100,1414.177500,1300.0,805.469147,1964166.120,1805583.780,MT,MT,MT,MT,iMports


In [5]:
cntComB = idb_sel.groupby(['COUNTRY_LABEL','COMMODITY_LABEL']).size().reset_index().rename(columns={0:'count'})
cntComB = cntComB[['COUNTRY_LABEL','COMMODITY_LABEL']]
cntComB.rename(columns={'COUNTRY_LABEL':'COUNTRY_IDB', 'COMMODITY_LABEL':'COMMODITY_IDB'}, inplace=True)
cntComB.head()

,COUNTRY_IDB,COMMODITY_IDB
0,BAHAMAS,Avocados
1,BAHAMAS,Bananas
2,BAHAMAS,Grapefruit
3,BAHAMAS,Mangoes
4,BAHAMAS,Onions


In [6]:
country_mapping = {    
    'BARBADOS':'BRB',
    'BAHAMAS':'BHS',
    'BOLIVIA':'BOL',
    'BELIZE':'BLZ',
    'ECUADOR':'ECU',
    'GUATEMALA':'GTM',
    'GUYANA':'GUY',
    'HONDURAS':'HND',
    'HAITI':'HTI',
    'JAMAICA':'JAM',
    'PERU':'PER',
    'NICARAGUA':'NIC',
    'PANAMA':'PAN',
    'PARAGUAY':'PRY',
    'SURINAME':'SUR',
    'EL SALVADOR':'SLV',
    'TRINIDAD AND TOBAGO':'TTO',
    'URUGUAY':'URY',
    'DOMINICAN REPUBLIC':'DOM'
}

currency_mapping = {    
    'BRB':'BBD',
    'BHS':'BSD',
    'BOL':'BOB',
    'BLZ':'BZD',
    'ECU':'USD',
    'GTM':'GTQ',
    'GUY':'GYD',
    'HND':'HNL',
    'HTI':'HTG',
    'JAM':'JMD',
    'PER':'PEN',
    'NIC':'NIO',
    'PAN':'PAB',
    'PRY':'PYG',
    'SUR':'SRD',
    'SLV':'SVC',
    'TTO':'TTD',
    'URY':'UYU',
    'DOM':'DOP'
}

In [7]:
idb_sel['COUNTRY_CODE'] = idb_sel.COUNTRY_LABEL.map(country_mapping)
idb_sel['REFP_MON_UNIT'] = idb_sel.COUNTRY_CODE.map(currency_mapping)
idb_sel['PROP_MON_UNIT'] = idb_sel.COUNTRY_CODE.map(currency_mapping)

In [8]:
date = datetime.strftime(datetime.today(),"%m/%d/%Y")
idb_sel['TIMESTAMP'] = date
idb_sel['SOURCE_FILE'] = 'IDB_Agrimonitor_-_PSE_Agricultural_Policy_Monitoring_System__data_11182024_Cat_1-3.xlsx'
idb_sel['SOURCE'] = 'IDB'

In [9]:
idb_sel.YEAR.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021, 2022, 2006, 2007, 2008, 2009, 2023], dtype=int64)

In [10]:
ex_rate = pd.read_csv(os.path.join(base_dir,"ExchangeRates.csv"))

ex_rate.set_index('country', inplace=True)
ex_rate = pd.DataFrame(ex_rate.unstack().reset_index())
ex_rate.rename(columns={'level_0':'YEAR', 'country':'COUNTRY_LABEL', 0:'ER_OFFICIAL'}, inplace=True)
ex_rate['COUNTRY_LABEL'] = ex_rate['COUNTRY_LABEL'].str.strip()
ex_rate = ex_rate[~ex_rate['COUNTRY_LABEL'].isin(countries)]
ex_rate = ex_rate[~ex_rate['COUNTRY_LABEL'].isin(['Canada','OECD total'])]

ex_rate['EROFFICIAL_SOURCE']='IDB'
ex_rate.YEAR = ex_rate.YEAR.astype(int)
ex_rate = ex_rate[ex_rate['YEAR']>=2006]
ex_rate = ex_rate.reset_index(drop=True)
ex_rate.head()

,YEAR,COUNTRY_LABEL,ER_OFFICIAL,EROFFICIAL_SOURCE
0,2006,BAHAMAS,NaN,IDB
1,2006,BARBADOS,NaN,IDB
2,2006,BELIZE,NaN,IDB
3,2006,BOLIVIA,8.011534,IDB
4,2006,DOMINICAN REPUBLIC,33.096400,IDB


In [11]:
idb_data = idb_sel.merge(ex_rate, on=['COUNTRY_LABEL','YEAR'])
idb_data['EROFFICIAL_UNIT'] = idb_data.COUNTRY_CODE.map(currency_mapping)

idb_data['NOTE_0'] = 0
idb_data['NOTE_1'] = np.nan
idb_data['NOTE_2'] = np.nan
idb_data['NOTE_3'] = np.nan
idb_data['NOTE_4'] = np.nan
idb_data['NOTE_9'] = np.nan


idb_data = idb_data[['COUNTRY_LABEL','COUNTRY_CODE','COMMODITY_LABEL','COMMODITY_CODE','YEAR','PROP','PROP_PHY_UNIT','REFP',
                     'REFP_PHY_UNIT','MPD_SOURCE','MPS','NPC_SOURCE','EFC','CONSQ','CONSQ_PHY_UNIT', 'PRODQ',
                     'PRODQ_PHY_UNIT','VC', 'VP','TRADE_STATUS','TIMESTAMP', 'SOURCE_FILE', 'SOURCE', 'PROP_MON_UNIT','REFP_MON_UNIT',\
                     'ER_OFFICIAL','EROFFICIAL_SOURCE','EROFFICIAL_UNIT', 'NOTE_0','NOTE_1','NOTE_2','NOTE_3','NOTE_4','NOTE_9']]
idb_data.shape

(2342, 34)

In [12]:
idb_data = idb_data[~( (idb_data['COUNTRY_LABEL']=='EL SALVADOR') & (idb_data['COMMODITY_LABEL']=='Eggs') 
                & (idb_data['YEAR'].isin([2009,2010])) )]

In [13]:
jm = idb_data[(idb_data['COUNTRY_LABEL']=='JAMAICA') & (idb_data['COMMODITY_LABEL']=='Refined Sugar') & 
              (idb_data['YEAR'].isin([2006,2007,2008,2009,2010,2011,2012,2013,2014]))]
jm['PRODQ'] = jm['PRODQ']*.08

idb_data = idb_data[~((idb_data['COUNTRY_LABEL']=='JAMAICA') & (idb_data['COMMODITY_LABEL']=='Refined Sugar')
                     & (idb_data['YEAR'].isin([2006,2007,2008,2009,2010,2011,2012,2013,2014])))]
idb_data = idb_data.append(jm)
print(idb_data.shape)


(2342, 34)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\4057331708.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jm['PRODQ'] = jm['PRODQ']*.08
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\4057331708.py:7: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.append(jm)


In [14]:
idb_data['IF(QP*PP=VP)'] = np.where(idb_data['PRODQ']*idb_data['PROP']==idb_data['VP'], 1, 0)
idb_data['IF(QP*PP~VP)'] = np.where(abs(idb_data['PRODQ']*idb_data['PROP']-idb_data['VP'])<.001, 1, 0)

In [15]:
vp_notequal = idb_data[idb_data['IF(QP*PP~VP)']==0]
vp_notequal['VP'] = vp_notequal['PRODQ']*vp_notequal['PROP']
print(vp_notequal.shape)

idb_data = idb_data[~(idb_data['IF(QP*PP~VP)']==0)]
print(idb_data.shape)
idb_data = idb_data.append(vp_notequal)
print(idb_data.shape)

# idb_data.ix[vp_notequal.index]=vp_notequal

(1804, 36)
(538, 36)
(2342, 36)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3032963077.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vp_notequal['VP'] = vp_notequal['PRODQ']*vp_notequal['PROP']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3032963077.py:7: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.append(vp_notequal)


In [16]:
idb_data['IF(QP*PP=VP)'] = np.where(idb_data['PRODQ']*idb_data['PROP']==idb_data['VP'], 1, 0)
idb_data['IF(QP*PP~VP)'] = np.where(abs(idb_data['PRODQ']*idb_data['PROP']-idb_data['VP'])<.001, 1, 0)

In [17]:
# Honey for Guatemala: Production quantity exists but no producer price for 2012-2017
vpnotequal = idb_data[idb_data['IF(QP*PP~VP)']==0]
vpnotequal.shape

(50, 36)

In [18]:
# CASE E
mps_gr_zero = idb_data['MPS'] > 0
mpd_ls_zero = idb_data['MPD_SOURCE'] < 0
efc_geq_zero = idb_data['EFC'] >=0
case_e = idb_data[mps_gr_zero & mpd_ls_zero & efc_geq_zero]
case_e['REFP'] = case_e.PROP-case_e.MPD_SOURCE
# case_e['NOTE_0'] = case_e['NOTE_0'].replace(0, np.nan)
# case_e['NOTE_3'] = "3"
# m.ix[case_e.index] = case_e
case_e.shape

(0, 36)

In [19]:
# idb_data['gap_PP_RP_MPD'] = np.where(abs(idb_data['PROP']-idb_data['REFP']-idb_data['MPD_SOURCE'])>.005*idb_data['PROP'], 1,0)
idb_data['gap_pp_rp_mpd'] = idb_data['PROP']-idb_data['REFP']-idb_data['MPD_SOURCE']
idb_data['MPD_neq_PP_RP'] = np.where(abs(idb_data['PROP']-idb_data['REFP']-idb_data['MPD_SOURCE'])>.005*idb_data['PROP'], 1,0)

In [20]:
idb_data.head()

,COUNTRY_LABEL,COUNTRY_CODE,COMMODITY_LABEL,COMMODITY_CODE,YEAR,PROP,PROP_PHY_UNIT,REFP,REFP_PHY_UNIT,MPD_SOURCE,...,NOTE_0,NOTE_1,NOTE_2,NOTE_3,NOTE_4,NOTE_9,IF(QP*PP=VP),IF(QP*PP~VP),gap_pp_rp_mpd,MPD_neq_PP_RP
0,BAHAMAS,BHS,Avocados,2,2010,815.709400,MT,687.195606,MT,128.513794,...,0,NaN,NaN,NaN,NaN,NaN,0,1,-2.842171e-14,0
1,BAHAMAS,BHS,Bananas,6,2010,925.940400,MT,681.050000,MT,244.890400,...,0,NaN,NaN,NaN,NaN,NaN,0,1,0.000000e+00,0
2,BAHAMAS,BHS,Grapefruit,17,2010,763.977228,MT,618.053795,MT,145.923433,...,0,NaN,NaN,NaN,NaN,NaN,0,1,0.000000e+00,0
3,BAHAMAS,BHS,Oranges,24,2010,925.940400,MT,591.221505,MT,334.718895,...,0,NaN,NaN,NaN,NaN,NaN,1,1,-1.136868e-13,0
6,BAHAMAS,BHS,Onions,55,2010,617.293600,MT,1886.370000,MT,0.000000,...,0,NaN,NaN,NaN,NaN,NaN,1,1,-1.269076e+03,1


In [21]:
mpd_zero = idb_data[(idb_data['MPD_neq_PP_RP']==1) & (idb_data['MPD_SOURCE']==0)]
print(mpd_zero.shape)

# Printing cases of MPD  detected zero
mpd_zero = mpd_zero[['COUNTRY_LABEL', 'COMMODITY_LABEL', 'YEAR']]

mpd_zero.rename(columns={'COUNTRY_LABEL':'COUNTRY'}, inplace=True)

mpd_zero = mpd_zero.groupby(['COUNTRY','COMMODITY_LABEL'])['YEAR'].apply(lambda x: list(np.unique(x)))
# mpd_zero = mpd_zero.groupby(['COUNTRY','COMMODITY_LABEL'])['YEAR'].size()
mpd_zero = mpd_zero.to_frame().reset_index()
# mpd_zero['YEAR'] = mpd_zero['YEAR'].astype(int)
mpd_zero = mpd_zero.pivot(index='COUNTRY', columns='COMMODITY_LABEL', values='YEAR').reset_index(col_level=1)
mpd_zero = mpd_zero.rename_axis(None, axis=1)

(582, 38)


In [22]:
mpd_zero.to_excel('IDB_MPD_Detected_Zero_Case.xlsx', na_rep='', index=False)
idb_data.shape

(2342, 38)

In [23]:
gap_mpd_zero = idb_data[(idb_data['MPD_neq_PP_RP']==1) & (idb_data['MPD_SOURCE']==0)]

sur_ban = gap_mpd_zero[((gap_mpd_zero['COMMODITY_LABEL']=='Bananas')& 
                             (gap_mpd_zero['COUNTRY_CODE']=='SUR')& (gap_mpd_zero['YEAR'].isin([2009,2010])))]

gap_mpd_zero= gap_mpd_zero[~((gap_mpd_zero['COMMODITY_LABEL']=='Bananas')& 
                             (gap_mpd_zero['COUNTRY_CODE']=='SUR')& (gap_mpd_zero['YEAR'].isin([2009,2010])))]
gap_mpd_zero['REFP'] = gap_mpd_zero['PROP']
gap_mpd_zero['NOTE_0'] = gap_mpd_zero['NOTE_0'].replace(0, np.nan)
gap_mpd_zero['NOTE_4'] = "4"
print(gap_mpd_zero.shape)

idb_data = idb_data[~((idb_data['MPD_neq_PP_RP']==1) & (idb_data['MPD_SOURCE']==0))]
print(idb_data.shape)
idb_data = idb_data.append(gap_mpd_zero)
idb_data = idb_data.append(sur_ban)
print(idb_data.shape)

(580, 38)
(1760, 38)
(2342, 38)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\2358245612.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.append(gap_mpd_zero)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\2358245612.py:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.append(sur_ban)


In [24]:
nic_sugar = idb_data[(idb_data['COUNTRY_LABEL']=='NICARAGUA') & (idb_data['COMMODITY_LABEL']=='Refined Sugar') & 
              (idb_data['YEAR']==2015)]
nic_sugar['REFP'] = nic_sugar['PROP']
nic_sugar['NOTE_0'] = nic_sugar['NOTE_0'].replace(0, np.nan)
nic_sugar['NOTE_4'] = "4"

idb_data = idb_data[~((idb_data['COUNTRY_LABEL']=='NICARAGUA') & (idb_data['COMMODITY_LABEL']=='Refined Sugar')
                     & (idb_data['YEAR']==2015))]
idb_data = idb_data.append(nic_sugar)
idb_data.shape


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3483597378.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nic_sugar['REFP'] = nic_sugar['PROP']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3483597378.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nic_sugar['NOTE_0'] = nic_sugar['NOTE_0'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3483597378.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

(2342, 38)

In [25]:
gap_mpd_gtzero = idb_data[(idb_data['MPD_neq_PP_RP']==1) & (idb_data['MPD_SOURCE']>0)]

gtm = gap_mpd_gtzero[(gap_mpd_gtzero['COMMODITY_LABEL']=='Refined Sugar') & (gap_mpd_gtzero['COUNTRY_CODE']=='GTM')]

gap_mpd_gtzero= gap_mpd_gtzero[~((gap_mpd_gtzero['COMMODITY_LABEL']=='Refined Sugar') & (gap_mpd_gtzero['COUNTRY_CODE']=='GTM'))]
gap_mpd_gtzero['REFP'] = gap_mpd_gtzero['PROP']-gap_mpd_gtzero['MPD_SOURCE']
gap_mpd_gtzero['NOTE_0'] = gap_mpd_gtzero['NOTE_0'].replace(0, np.nan)
gap_mpd_gtzero['NOTE_3'] = "3"
print(gap_mpd_gtzero.shape)

idb_data = idb_data[~((idb_data['MPD_neq_PP_RP']==1) & (idb_data['MPD_SOURCE']>0))]
print(idb_data.shape)
idb_data = idb_data.append(gap_mpd_gtzero)
print(idb_data.shape)
idb_data = idb_data.append(gtm)
print(idb_data.shape)

(6, 38)
(2323, 38)
(2329, 38)
(2342, 38)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\4158071173.py:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.append(gap_mpd_gtzero)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\4158071173.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.append(gtm)


In [26]:
gap_mpd_lszero = idb_data[(idb_data['MPD_neq_PP_RP']==1) & (idb_data['MPD_SOURCE']<0)]

In [27]:
idb_data['gap_pp_rp_mpd'] = idb_data['PROP']-idb_data['REFP']-idb_data['MPD_SOURCE']
idb_data['MPD_neq_PP_RP'] = np.where(abs(idb_data['PROP']-idb_data['REFP']-idb_data['MPD_SOURCE'])>.005*idb_data['PROP'], 1,0)

In [28]:
## Fixing Jamaica Sugar producer price issue. IDB provided data on refined equivalent producer price 
jm = idb_data[(idb_data['COUNTRY_LABEL']=='JAMAICA') & (idb_data['COMMODITY_LABEL']=='Refined Sugar')]

jm['NOTE_0'] = jm['NOTE_0'].replace(0, np.nan)
jm['NOTE_1'] = 1

idb_data = idb_data[~((idb_data['COUNTRY_LABEL']=='JAMAICA') & (idb_data['COMMODITY_LABEL']=='Refined Sugar'))]
print(idb_data.shape)
idb_data = idb_data.append(jm)
print(idb_data.shape)


(2328, 38)
(2342, 38)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\960142320.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jm['NOTE_0'] = jm['NOTE_0'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\960142320.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jm['NOTE_1'] = 1
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\960142320.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb_data.

In [29]:
gtm = idb_data[((idb_data['COMMODITY_LABEL']=='Refined Sugar') & (idb_data['COUNTRY_CODE']=='GTM'))]
gtm['NOTE_0'] = gtm['NOTE_0'].replace(0, np.nan)
gtm['NOTE_9'] = 9
print(gtm.shape)

idb_data = idb_data[~((idb_data['COMMODITY_LABEL']=='Refined Sugar') & (idb_data['COUNTRY_CODE']=='GTM'))]
print(idb_data.shape)
idb_data = idb_data.append(gtm)
print(idb_data.shape)

# idb_data.ix[gtm.index] = gtm

idb_data = idb_data[~((idb_data.COUNTRY_CODE=='GTM') & (idb_data.COMMODITY_LABEL=='Honey') & 
                      (idb_data.YEAR.isin([2012,2013,2014,2015,2016,2017])))]


(13, 38)
(2329, 38)
(2342, 38)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3605392292.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gtm['NOTE_0'] = gtm['NOTE_0'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3605392292.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gtm['NOTE_9'] = 9
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18392\3605392292.py:8: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_data = idb

In [30]:
idb_data = idb_data[['COUNTRY_LABEL','COUNTRY_CODE','COMMODITY_LABEL','COMMODITY_CODE','YEAR','PROP','PROP_PHY_UNIT','REFP','REFP_PHY_UNIT',
         'MPD_SOURCE', 'MPS','NPC_SOURCE','EFC','CONSQ','CONSQ_PHY_UNIT', 'PRODQ','PRODQ_PHY_UNIT','VC','VP','TRADE_STATUS','TIMESTAMP',
         'SOURCE_FILE','SOURCE','PROP_MON_UNIT','REFP_MON_UNIT','ER_OFFICIAL','EROFFICIAL_SOURCE','EROFFICIAL_UNIT',
         'NOTE_0','NOTE_1', 'NOTE_2','NOTE_3', 'NOTE_4','NOTE_9']]

idb_data.to_csv('IDB_input_file_2024.csv', index=False)